<a href="https://colab.research.google.com/github/Pushkarsinghs/indian_stock-analysis/blob/main/09_streamlit_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 NIFTY 50 Intelligence System
## Notebook 9: Streamlit Web Application

Builds and runs a live interactive web app powered by all
the analysis from Notebooks 1-8.

### Phase 1: Test locally via Colab + ngrok (temporary URL)
### Phase 2: Deploy to Streamlit Cloud (permanent free URL)

### Pages:
1. Market Overview
2. Stock Deep Dive
3. Fundamental Analysis
4. Portfolio & Risk
5. Price Forecast
6. Strategy Backtest
7. Sentiment Intelligence (FinBERT)

## ✅ Streamlit App Running!

### Test in Colab:
Run Cell 15 → get ngrok URL → open in browser

### Deploy Permanently to Streamlit Cloud (Phase 2):
1. Push streamlit_app/ folder to GitHub
2. Go to share.streamlit.io
3. Connect GitHub → select repo
4. Main file: app.py
5. Deploy → get permanent URL

### Your 7-page app covers:
1. app.py              → Home / Market Overview
2. 02_Stock_Deep_Dive  → Technical charts (Plotly)
3. 03_Fundamental      → Scores, ROE, grades
4. 04_Portfolio_Risk   → Allocation, Sharpe, VaR
5. 05_Price_Forecast   → Prophet predictions
6. 06_Strategy_Backtest→ Signal vs Buy & Hold
7. 07_Sentiment        → FinBERT NLP analysis

In [16]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

BASE       = '/content/drive/MyDrive/indian_stock_analysis'
APP_DIR    = '/content/streamlit_app'
LOCAL_DATA = '/content/streamlit_data'

# Create all required folders
for folder in [
    APP_DIR,
    f"{APP_DIR}/pages",
    f"{APP_DIR}/.streamlit",
    LOCAL_DATA
]:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ Created: {folder}")

print("\n📋 Copying CSV files from Drive to local storage...")

FILES_NEEDED = [
    "nifty50_for_powerbi.csv",
    "nifty50_technical_powerbi.csv",
    "latest_signals.csv",
    "nifty50_fundamentals_powerbi.csv",
    "nifty50_sentiment_powerbi.csv",
    "nifty50_headlines_powerbi.csv",
    "nifty50_forecasts_powerbi.csv",
    "forecast_summary_powerbi.csv",
    "nifty50_risk_metrics_powerbi.csv",
    "portfolio_allocation_powerbi.csv",
    "portfolio_performance_powerbi.csv",
    "backtest_equity_powerbi.csv",
    "backtest_trades_powerbi.csv",
    "backtest_summary_powerbi.csv",
    "forecast_accuracy_powerbi.csv",
    "forecast_mape_summary_powerbi.csv",
]

for fname in FILES_NEEDED:
    src = f"{BASE}/data/output/{fname}"
    dst = f"{LOCAL_DATA}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst) / 1024
        print(f"  ✅ {fname:<45} {size:>8.1f} KB")
    else:
        print(f"  ⚠️  {fname} — not found in Drive")

print("\n✅ All folders and data ready!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Created: /content/streamlit_app
✅ Created: /content/streamlit_app/pages
✅ Created: /content/streamlit_app/.streamlit
✅ Created: /content/streamlit_data

📋 Copying CSV files from Drive to local storage...
  ✅ nifty50_for_powerbi.csv                         1433.9 KB
  ✅ nifty50_technical_powerbi.csv                   5341.1 KB
  ✅ latest_signals.csv                                23.1 KB
  ✅ nifty50_fundamentals_powerbi.csv                  10.5 KB
  ✅ nifty50_sentiment_powerbi.csv                      3.3 KB
  ✅ nifty50_headlines_powerbi.csv                     69.4 KB
  ✅ nifty50_forecasts_powerbi.csv                    603.4 KB
  ✅ forecast_summary_powerbi.csv                       3.5 KB
  ✅ nifty50_risk_metrics_powerbi.csv                   4.2 KB
  ✅ portfolio_allocation_powerbi.csv                   0.5 KB
  ✅ portfolio_performance_powerbi.csv        

In [17]:

APP_DRIVE = f"{BASE}/streamlit_app"

if os.path.exists(APP_DRIVE):
    print("✅ Found saved app in Google Drive — restoring...")
    for root, dirs, files in os.walk(APP_DRIVE):
        for file in files:
            src = os.path.join(root, file)
            rel = os.path.relpath(src, APP_DRIVE)
            dst = os.path.join(APP_DIR, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            print(f"  ✅ Restored: {rel}")
    print("\n✅ App restored from Drive!")
    print("Skip to Cell 8 (Launch App) — no need to rewrite files")
else:
    print("⚠️  No saved app found in Drive")
    print("Continue with Cells 3-7 to recreate all files")

✅ Found saved app in Google Drive — restoring...
  ✅ Restored: requirements.txt
  ✅ Restored: data_loader.py
  ✅ Restored: app.py
  ✅ Restored: pages/07_Sentiment_Intelligence.py
  ✅ Restored: pages/03_Fundamental_Analysis.py
  ✅ Restored: pages/02_Stock_Deep_Dive.py
  ✅ Restored: pages/06_Strategy_Backtest.py
  ✅ Restored: pages/04_Portfolio_Risk.py
  ✅ Restored: pages/05_Price_Forecast.py
  ✅ Restored: .streamlit/config.toml
  ✅ Restored: __pycache__/data_loader.cpython-312.pyc

✅ App restored from Drive!
Skip to Cell 8 (Launch App) — no need to rewrite files


In [18]:
config_content = """
[theme]
primaryColor         = "#1F77B4"
backgroundColor      = "#FFFFFF"
secondaryBackgroundColor = "#F0F2F5"
textColor            = "#1F3864"
font                 = "sans serif"

[server]
headless = true
port     = 8501
"""

with open(f"{APP_DIR}/.streamlit/config.toml", "w") as f:
    f.write(config_content)

print("✅ Config created")

✅ Config created


In [19]:
data_loader_code = '''
import pandas as pd
import streamlit as st

DATA_DIR = "/content/streamlit_data"

@st.cache_data(ttl=3600)
def load_technical():
    return pd.read_csv(f"{DATA_DIR}/nifty50_technical_powerbi.csv", parse_dates=["Date"])

@st.cache_data(ttl=3600)
def load_signals():
    return pd.read_csv(f"{DATA_DIR}/latest_signals.csv")

@st.cache_data(ttl=3600)
def load_fundamentals():
    return pd.read_csv(f"{DATA_DIR}/nifty50_fundamentals_powerbi.csv")

@st.cache_data(ttl=3600)
def load_sentiment():
    return pd.read_csv(f"{DATA_DIR}/nifty50_sentiment_powerbi.csv")

@st.cache_data(ttl=3600)
def load_headlines():
    return pd.read_csv(f"{DATA_DIR}/nifty50_headlines_powerbi.csv")

@st.cache_data(ttl=3600)
def load_forecasts():
    return pd.read_csv(f"{DATA_DIR}/nifty50_forecasts_powerbi.csv", parse_dates=["Date"])

@st.cache_data(ttl=3600)
def load_forecast_summary():
    return pd.read_csv(f"{DATA_DIR}/forecast_summary_powerbi.csv")

@st.cache_data(ttl=3600)
def load_risk_metrics():
    return pd.read_csv(f"{DATA_DIR}/nifty50_risk_metrics_powerbi.csv")

@st.cache_data(ttl=3600)
def load_portfolio_allocation():
    return pd.read_csv(f"{DATA_DIR}/portfolio_allocation_powerbi.csv")

@st.cache_data(ttl=3600)
def load_portfolio_performance():
    return pd.read_csv(f"{DATA_DIR}/portfolio_performance_powerbi.csv")

@st.cache_data(ttl=3600)
def load_backtest_equity():
    return pd.read_csv(f"{DATA_DIR}/backtest_equity_powerbi.csv", parse_dates=["Date"])

@st.cache_data(ttl=3600)
def load_backtest_trades():
    return pd.read_csv(f"{DATA_DIR}/backtest_trades_powerbi.csv", parse_dates=["Date"])

@st.cache_data(ttl=3600)
def load_backtest_summary():
    return pd.read_csv(f"{DATA_DIR}/backtest_summary_powerbi.csv")

@st.cache_data(ttl=3600)
def load_forecast_accuracy():
    return pd.read_csv(f"{DATA_DIR}/forecast_accuracy_powerbi.csv")

@st.cache_data(ttl=3600)
def load_mape_summary():
    return pd.read_csv(f"{DATA_DIR}/forecast_mape_summary_powerbi.csv")
'''

with open(f"{APP_DIR}/data_loader.py", "w") as f:
    f.write(data_loader_code)

print("✅ data_loader.py created")

✅ data_loader.py created


In [20]:
app_code = '''
import streamlit as st
import pandas as pd
import sys
sys.path.append("/content/streamlit_app")

from data_loader import (
    load_signals, load_sentiment, load_risk_metrics,
    load_backtest_summary, load_backtest_trades
)

st.set_page_config(
    page_title="NIFTY 50 Intelligence System",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    .main-header {
        background: linear-gradient(135deg, #1F3864, #2E5EAA);
        padding: 25px 30px; border-radius: 12px;
        color: white; text-align: center;
        margin-bottom: 25px;
        box-shadow: 0 4px 15px rgba(31,56,100,0.3);
    }
    .main-header h1 { font-size: 2.2rem; margin: 0; font-weight: 800; }
    .main-header p  { font-size: 0.95rem; opacity: 0.85; margin: 6px 0 0 0; }
    .metric-card {
        background: linear-gradient(135deg, #1F3864, #2E5EAA);
        padding: 20px 15px; border-radius: 10px; color: white;
        text-align: center; box-shadow: 0 3px 10px rgba(0,0,0,0.2);
        margin-bottom: 10px;
    }
    .metric-value { font-size: 2.2rem; font-weight: 800; line-height: 1.1; }
    .metric-label {
        font-size: 0.75rem; opacity: 0.85; margin-top: 5px;
        text-transform: uppercase; letter-spacing: 0.8px;
    }
    .section-header {
        color: #1F3864; font-size: 1.1rem; font-weight: 700;
        border-bottom: 2px solid #1F77B4;
        padding-bottom: 6px; margin: 20px 0 15px 0;
    }
    [data-testid="stSidebar"] { background: #F0F2F5; }
    .block-container { padding-top: 1rem; }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="main-header">
    <h1>📈 NIFTY 50 Intelligence System</h1>
    <p>End-to-end automated stock market analysis for all 50 NIFTY 50 stocks</p>
    <p>Python · FinBERT NLP · Prophet Forecasting · Portfolio Optimization · Power BI</p>
</div>
""", unsafe_allow_html=True)

try:
    signals  = load_signals()
    sent     = load_sentiment()
    risk     = load_risk_metrics()
    backtest = load_backtest_summary()
    trades   = load_backtest_trades()
except Exception as e:
    st.error(f"Error loading data: {e}")
    st.stop()

bull = len(signals[signals["Signal"].isin(["Strong Buy","Buy","Weak Buy"])]) if "Signal" in signals.columns else 0
bear = len(signals[signals["Signal"].isin(["Strong Sell","Sell","Weak Sell"])]) if "Signal" in signals.columns else 0
sent_score   = round(sent["Sentiment_Score"].mean(), 1) if not sent.empty and "Sentiment_Score" in sent.columns else 0
win_rate     = round((backtest["Beat_Benchmark"]=="Yes").mean()*100, 1) if not backtest.empty and "Beat_Benchmark" in backtest.columns else 0
avg_sharpe   = round(risk["Sharpe_Ratio"].mean(), 3) if not risk.empty and "Sharpe_Ratio" in risk.columns else 0
total_trades = len(trades) if not trades.empty else 0

st.markdown('<p class="section-header">📊 Live Market Snapshot</p>', unsafe_allow_html=True)

col1,col2,col3,col4,col5 = st.columns(5)
for col, value, color, label in [
    (col1, "49",           "#FFFFFF", "Stocks Tracked"),
    (col2, str(bull),      "#90EE90", "🟢 Bullish Signals"),
    (col3, str(bear),      "#FF6B6B", "🔴 Bearish Signals"),
    (col4, str(sent_score),"#FFD700", "💬 Market Sentiment"),
    (col5, str(avg_sharpe),"#90EE90", "📊 Avg Sharpe Ratio"),
]:
    with col:
        st.markdown(f"""
        <div class="metric-card">
            <div class="metric-value" style="color:{color}">{value}</div>
            <div class="metric-label">{label}</div>
        </div>""", unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

col_left, col_right = st.columns(2)
with col_left:
    st.markdown('<p class="section-header">🟢 Top 5 Gainers Today</p>', unsafe_allow_html=True)
    if "Daily_Return" in signals.columns:
        gainers = signals.nlargest(5,"Daily_Return")[["Ticker","Close","Daily_Return","Signal"]].copy()
        gainers["Daily_Return"] = (gainers["Daily_Return"]*100).round(2).astype(str)+"%"
        gainers.columns = ["Ticker","Price (Rs)","Return","Signal"]
        st.dataframe(gainers, use_container_width=True, hide_index=True)

with col_right:
    st.markdown('<p class="section-header">🔴 Top 5 Losers Today</p>', unsafe_allow_html=True)
    if "Daily_Return" in signals.columns:
        losers = signals.nsmallest(5,"Daily_Return")[["Ticker","Close","Daily_Return","Signal"]].copy()
        losers["Daily_Return"] = (losers["Daily_Return"]*100).round(2).astype(str)+"%"
        losers.columns = ["Ticker","Price (Rs)","Return","Signal"]
        st.dataframe(losers, use_container_width=True, hide_index=True)

st.markdown('<p class="section-header">⭐ Strong Buy Signals Right Now</p>', unsafe_allow_html=True)
if "Signal" in signals.columns:
    strong_buy = signals[signals["Signal"]=="Strong Buy"][["Ticker","Close","RSI","Signal_Score","Signal"]].copy()
    if not strong_buy.empty:
        st.dataframe(strong_buy.sort_values("Signal_Score",ascending=False), use_container_width=True, hide_index=True)
    else:
        st.info("No Strong Buy signals currently")

st.markdown('<p class="section-header">📊 Strategy Backtest Summary</p>', unsafe_allow_html=True)
col_b1,col_b2,col_b3 = st.columns(3)
with col_b1: st.metric("Strategy Win Rate", f"{win_rate}%")
with col_b2: st.metric("Total Trades Executed", f"{total_trades:,}")
with col_b3:
    best = backtest.nlargest(1,"Outperformance_Pct")["Ticker"].values[0].replace(".NS","") if not backtest.empty and "Outperformance_Pct" in backtest.columns else "N/A"
    st.metric("Best Signal Stock", best)

st.markdown('<p class="section-header">💬 FinBERT Sentiment Snapshot</p>', unsafe_allow_html=True)
if not sent.empty and "Sentiment_Label" in sent.columns:
    cs1,cs2 = st.columns(2)
    with cs1:
        st.subheader("🟢 Most Bullish ")
        bull_sent = sent[sent["Sentiment_Label"].isin(["Positive","Very Positive"])]
        if not bull_sent.empty:
            st.dataframe(bull_sent.nlargest(5,"Sentiment_Score")[["Ticker","Sentiment_Label","Sentiment_Score","Avg_Confidence"]], use_container_width=True, hide_index=True)
    with cs2:
        st.subheader("🔴 Most Bearish ")
        bear_sent = sent[sent["Sentiment_Label"].isin(["Negative","Very Negative"])]
        if not bear_sent.empty:
            st.dataframe(bear_sent.nsmallest(5,"Sentiment_Score")[["Ticker","Sentiment_Label","Sentiment_Score","Avg_Confidence"]], use_container_width=True, hide_index=True)

st.markdown("---")
st.markdown("""
**📈 NIFTY 50 Intelligence System** | Built by **Pushkar Singh** |
Python · FinBERT · Prophet · LSTM · PyPortfolioOpt · Power BI |
[GitHub](https://github.com/Pushkarsinghs/indian_stock-analysis.git)
""")
'''

with open(f"{APP_DIR}/app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created")

✅ app.py created


In [21]:
# ── Page 2: Stock Deep Dive ──
page2 = '''
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
sys.path.append("/content/streamlit_app")
from data_loader import load_technical, load_signals

st.set_page_config(page_title="Stock Deep Dive", page_icon="🔍", layout="wide")
st.markdown("""<style>[data-testid="stSidebar"]{background:#F0F2F5;}.block-container{padding-top:1rem;}</style>""", unsafe_allow_html=True)
st.markdown("""<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">🔍 Stock Deep Dive — Technical Analysis</h2></div>""", unsafe_allow_html=True)

df      = load_technical()
signals = load_signals()
df["Date"] = pd.to_datetime(df["Date"])

st.sidebar.header("Controls")
ticker = st.sidebar.selectbox("Select Stock", sorted(df["Ticker"].unique()), index=0)
period = st.sidebar.selectbox("Date Range", ["1 Month","3 Months","6 Months","1 Year"], index=3)
show_bb  = st.sidebar.checkbox("Bollinger Bands", value=True)
show_sma = st.sidebar.checkbox("Moving Averages", value=True)

stock  = df[df["Ticker"]==ticker].copy().sort_values("Date")
days   = {"1 Month":30,"3 Months":90,"6 Months":180,"1 Year":365}[period]
cutoff = stock["Date"].max() - pd.Timedelta(days=days)
stock  = stock[stock["Date"] >= cutoff]

latest     = stock.iloc[-1]
signal_row = signals[signals["Ticker"]==ticker]
signal_val = signal_row["Signal"].values[0] if not signal_row.empty else "N/A"

c1,c2,c3,c4,c5 = st.columns(5)
c1.metric("Price",  f"Rs{latest[\'Close\']:,.2f}")
c2.metric("RSI",    f"{latest[\'RSI\']:.1f}")
c3.metric("MACD",   f"{latest[\'MACD\']:.2f}")
c4.metric("Signal", signal_val)
c5.metric("Return", f"{latest[\'Daily_Return\']*100:.2f}%", delta=f"{latest[\'Daily_Return\']*100:.2f}%")

fig = make_subplots(rows=4,cols=1,shared_xaxes=True,vertical_spacing=0.04,
    subplot_titles=[f"{ticker} Price","Volume","RSI (14)","MACD"],
    row_heights=[0.45,0.15,0.20,0.20])

fig.add_trace(go.Scatter(x=stock["Date"],y=stock["Close"],name="Close",line=dict(color="#1F77B4",width=2)),row=1,col=1)
if show_sma:
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["SMA_20"],name="SMA 20",line=dict(color="#FF7F0E",width=1.5,dash="dash")),row=1,col=1)
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["SMA_50"],name="SMA 50",line=dict(color="#2CA02C",width=1.5,dash="dash")),row=1,col=1)
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["EMA_20"],name="EMA 20",line=dict(color="#9467BD",width=1,dash="dot")),row=1,col=1)
if show_bb:
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["BB_Upper"],name="BB Upper",line=dict(color="#D62728",width=1,dash="dot"),showlegend=False),row=1,col=1)
    fig.add_trace(go.Scatter(x=stock["Date"],y=stock["BB_Lower"],name="BB Lower",line=dict(color="#2CA02C",width=1,dash="dot"),fill="tonexty",fillcolor="rgba(128,128,128,0.08)",showlegend=False),row=1,col=1)

vol_colors = ["#2CA02C" if r>=0 else "#D62728" for r in stock["Daily_Return"].fillna(0)]
fig.add_trace(go.Bar(x=stock["Date"],y=stock["Volume"],name="Volume",marker_color=vol_colors,showlegend=False),row=2,col=1)
fig.add_trace(go.Scatter(x=stock["Date"],y=stock["RSI"],name="RSI",line=dict(color="#9467BD",width=1.5)),row=3,col=1)
fig.add_hline(y=70,line_dash="dash",line_color="#D62728",annotation_text="Overbought",row=3,col=1)
fig.add_hline(y=30,line_dash="dash",line_color="#2CA02C",annotation_text="Oversold",row=3,col=1)
fig.add_trace(go.Scatter(x=stock["Date"],y=stock["MACD"],name="MACD",line=dict(color="#1F77B4",width=1.5)),row=4,col=1)
fig.add_trace(go.Scatter(x=stock["Date"],y=stock["MACD_Signal"],name="Signal",line=dict(color="#D62728",width=1.5,dash="dash")),row=4,col=1)
macd_colors = ["#2CA02C" if v>=0 else "#D62728" for v in stock["MACD_Hist"].fillna(0)]
fig.add_trace(go.Bar(x=stock["Date"],y=stock["MACD_Hist"],name="Hist",marker_color=macd_colors,showlegend=False),row=4,col=1)

fig.update_layout(height=750,template="plotly_white",legend=dict(orientation="h",y=1.02),xaxis_rangeslider_visible=False,margin=dict(t=40,b=20))
fig.update_yaxes(title_text="Price (Rs)",row=1,col=1)
fig.update_yaxes(title_text="Volume",row=2,col=1)
fig.update_yaxes(title_text="RSI",range=[0,100],row=3,col=1)
fig.update_yaxes(title_text="MACD",row=4,col=1)
st.plotly_chart(fig, use_container_width=True)
'''

with open(f"{APP_DIR}/pages/02_Stock_Deep_Dive.py", "w") as f:
    f.write(page2)
print("✅ Page 2 created")

# ── Page 3: Fundamental Analysis ──
page3 = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.append("/content/streamlit_app")
from data_loader import load_fundamentals

st.set_page_config(page_title="Fundamental Analysis", page_icon="📊", layout="wide")
st.markdown("""<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>""", unsafe_allow_html=True)
st.markdown("""<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">📊 Fundamental Analysis</h2></div>""", unsafe_allow_html=True)

fund = load_fundamentals()
st.sidebar.header("Filters")
sector_filter = st.sidebar.multiselect("Sector", sorted(fund["Sector"].dropna().unique()), default=[])
grade_filter  = st.sidebar.multiselect("Grade",  ["A","B","C","D","F"], default=[])
filtered = fund.copy()
if sector_filter: filtered = filtered[filtered["Sector"].isin(sector_filter)]
if grade_filter:  filtered = filtered[filtered["Fund_Grade"].isin(grade_filter)]

c1,c2,c3,c4 = st.columns(4)
c1.metric("Stocks", len(filtered))
c2.metric("Grade A/B", len(filtered[filtered["Fund_Grade"].isin(["A","B"])]))
c3.metric("Avg P/E", f"{filtered[\'PE_Ratio\'].mean():.1f}x" if "PE_Ratio" in filtered.columns else "N/A")
c4.metric("Avg ROE", f"{filtered[\'ROE_Pct\'].mean():.1f}%" if "ROE_Pct" in filtered.columns else "N/A")

st.markdown("---")
grade_colors = {"A":"#1A7A1A","B":"#2CA02C","C":"#FF7F0E","D":"#D62728","F":"#8B0000"}
cl,cr = st.columns(2)
with cl:
    st.subheader("Fundamental Score")
    top20 = filtered.nlargest(20,"Fund_Score")
    fig1  = px.bar(top20.sort_values("Fund_Score"),x="Fund_Score",y="Ticker",color="Fund_Grade",color_discrete_map=grade_colors,orientation="h")
    fig1.add_vline(x=50,line_dash="dash",line_color="gray")
    fig1.update_layout(height=500,template="plotly_white")
    st.plotly_chart(fig1, use_container_width=True)
with cr:
    st.subheader("ROE vs P/E Ratio")
    valid = filtered.dropna(subset=["PE_Ratio","ROE_Pct"])
    valid = valid[valid["PE_Ratio"].between(0,80)]
    fig2  = px.scatter(valid,x="PE_Ratio",y="ROE_Pct",size="Market_Cap_Cr" if "Market_Cap_Cr" in valid.columns else None,color="Sector",hover_data=["Ticker","Fund_Grade"])
    fig2.add_vline(x=25,line_dash="dash",line_color="gray",annotation_text="Fair Value")
    fig2.add_hline(y=15,line_dash="dash",line_color="gray",annotation_text="ROE Benchmark")
    fig2.update_layout(height=500,template="plotly_white")
    st.plotly_chart(fig2, use_container_width=True)

cl2,cr2 = st.columns(2)
with cl2:
    st.subheader("Grade Distribution")
    grade_counts = filtered["Fund_Grade"].value_counts().reset_index()
    grade_counts.columns = ["Grade","Count"]
    fig3 = px.pie(grade_counts,names="Grade",values="Count",color="Grade",color_discrete_map=grade_colors,hole=0.5)
    st.plotly_chart(fig3, use_container_width=True)
with cr2:
    st.subheader("Sector Average ROE")
    sector_roe = filtered.groupby("Sector")["ROE_Pct"].mean().reset_index().sort_values("ROE_Pct")
    fig4 = px.bar(sector_roe,x="ROE_Pct",y="Sector",orientation="h",color="ROE_Pct",color_continuous_scale=["#D62728","#FF7F0E","#2CA02C"])
    fig4.add_vline(x=15,line_dash="dash",line_color="navy",annotation_text="Benchmark")
    fig4.update_layout(height=400,template="plotly_white")
    st.plotly_chart(fig4, use_container_width=True)

st.subheader("Full Scorecard")
display_cols = [c for c in ["Ticker","Company","Sector","PE_Ratio","PB_Ratio","ROE_Pct","Profit_Margin_Pct","Debt_To_Equity","Dividend_Yield_Pct","Fund_Score","Fund_Grade"] if c in filtered.columns]
st.dataframe(filtered[display_cols].sort_values("Fund_Score",ascending=False) if "Fund_Score" in filtered.columns else filtered[display_cols], use_container_width=True, hide_index=True)
'''

with open(f"{APP_DIR}/pages/03_Fundamental_Analysis.py", "w") as f:
    f.write(page3)
print("✅ Page 3 created")

# ── Page 4: Portfolio & Risk ──
page4 = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.append("/content/streamlit_app")
from data_loader import load_risk_metrics, load_portfolio_allocation, load_portfolio_performance

st.set_page_config(page_title="Portfolio & Risk", page_icon="💼", layout="wide")
st.markdown("""<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>""", unsafe_allow_html=True)
st.markdown("""<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">💼 Portfolio & Risk Analysis</h2></div>""", unsafe_allow_html=True)

risk  = load_risk_metrics()
alloc = load_portfolio_allocation()
perf  = load_portfolio_performance()

c1,c2,c3,c4 = st.columns(4)
c1.metric("Portfolio Value", f"Rs{alloc[\'Value_INR\'].sum():,.0f}" if "Value_INR" in alloc.columns else "N/A")
c2.metric("Avg Sharpe", f"{risk[\'Sharpe_Ratio\'].mean():.3f}" if "Sharpe_Ratio" in risk.columns else "N/A")
c3.metric("Best Sharpe", risk.nlargest(1,"Sharpe_Ratio")["Ticker"].values[0].replace(".NS","") if not risk.empty else "N/A")
c4.metric("Stocks in Portfolio", len(alloc))

st.markdown("---")
cl,cr = st.columns(2)
with cl:
    st.subheader("Portfolio Allocation")
    if not alloc.empty and "Value_INR" in alloc.columns:
        fig1 = px.pie(alloc,names="Company",values="Value_INR",hole=0.3,title=f"Max Sharpe — Rs{alloc[\'Value_INR\'].sum():,.0f}")
        fig1.update_traces(textposition="outside",textinfo="label+percent")
        st.plotly_chart(fig1, use_container_width=True)
        st.dataframe(alloc[["Company","Shares","Price","Value_INR","Weight_Pct"]].sort_values("Value_INR",ascending=False), use_container_width=True, hide_index=True)
with cr:
    st.subheader("Risk vs Return")
    fig2 = px.scatter(risk,x="Ann_Volatility_Pct",y="Ann_Return_Pct",color="Sharpe_Ratio",color_continuous_scale=["#D62728","#FFFFFF","#2CA02C"],hover_data=["Ticker"],text="Ticker",labels={"Ann_Volatility_Pct":"Volatility (%)","Ann_Return_Pct":"Return (%)"})
    fig2.add_vline(x=20,line_dash="dash",line_color="#D62728",annotation_text="High Risk")
    fig2.add_hline(y=0,line_dash="dash",line_color="black")
    fig2.update_traces(textposition="top center",textfont_size=7)
    fig2.update_layout(height=450,template="plotly_white")
    st.plotly_chart(fig2, use_container_width=True)

st.subheader("Sharpe Ratio by Stock")
sharpe_sorted = risk.sort_values("Sharpe_Ratio",ascending=True)
colors = ["#2CA02C" if v>=0 else "#D62728" for v in sharpe_sorted["Sharpe_Ratio"]]
fig3 = go.Figure(go.Bar(x=sharpe_sorted["Sharpe_Ratio"],y=sharpe_sorted["Ticker"].str.replace(".NS",""),orientation="h",marker_color=colors))
fig3.add_vline(x=1.0,line_dash="dash",line_color="navy",annotation_text="Good Sharpe")
fig3.add_vline(x=0,line_color="black",line_width=0.8)
fig3.update_layout(height=600,template="plotly_white",xaxis_title="Sharpe Ratio")
st.plotly_chart(fig3, use_container_width=True)

if not perf.empty:
    st.subheader("Portfolio Strategy Comparison")
    st.dataframe(perf, use_container_width=True, hide_index=True)
'''

with open(f"{APP_DIR}/pages/04_Portfolio_Risk.py", "w") as f:
    f.write(page4)
print("✅ Page 4 created")

# ── Page 5: Price Forecast ──
page5 = '''
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import sys
sys.path.append("/content/streamlit_app")
from data_loader import load_forecasts, load_forecast_summary, load_sentiment

st.set_page_config(page_title="Price Forecast", page_icon="🔮", layout="wide")
st.markdown("""<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>""", unsafe_allow_html=True)
st.markdown("""<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">🔮 Price Forecast — 30-Day Prophet Predictions</h2></div>""", unsafe_allow_html=True)

try:
    forecasts = load_forecasts()
    summary   = load_forecast_summary()
    sent      = load_sentiment()
except Exception as e:
    st.error(f"Error loading data: {e}")
    st.stop()

forecasts["Date"] = pd.to_datetime(forecasts["Date"])
today_str = pd.Timestamp.today().strftime("%Y-%m-%d")
today_ts  = pd.Timestamp.today()

st.sidebar.header("Controls")
ticker = st.sidebar.selectbox("Select Stock", sorted(forecasts["Ticker"].unique()), index=0)

bull_fc    = len(summary[summary["Expected_Change"]>0]) if "Expected_Change" in summary.columns else 0
best_gain  = summary["Expected_Change"].max() if "Expected_Change" in summary.columns else 0
worst_loss = summary["Expected_Change"].min() if "Expected_Change" in summary.columns else 0

c1,c2,c3 = st.columns(3)
c1.metric("Bullish Forecasts", f"{bull_fc} stocks")
c2.metric("Best Expected Gain", f"+{best_gain:.1f}%")
c3.metric("Worst Expected Loss", f"{worst_loss:.1f}%")

st.markdown("---")
st.subheader(f"{ticker.replace('.NS','')} — 30-Day Forecast")
stock_fc   = forecasts[forecasts["Ticker"]==ticker].sort_values("Date")
historical = stock_fc[stock_fc["Date"]<=today_ts]
future     = stock_fc[stock_fc["Date"]>today_ts]

fig = go.Figure()
if not historical.empty:
    fig.add_trace(go.Scatter(x=historical["Date"],y=historical["Forecast"],name="Historical Fitted",line=dict(color="#1F77B4",width=1.5)))
if not future.empty:
    fig.add_trace(go.Scatter(x=future["Date"],y=future["Forecast"],name="30-Day Forecast",line=dict(color="#2CA02C",width=2.5)))
    fig.add_trace(go.Scatter(
        x=pd.concat([future["Date"],future["Date"].iloc[::-1]]),
        y=pd.concat([future["Upper_CI"],future["Lower_CI"].iloc[::-1]]),
        fill="toself",fillcolor="rgba(44,160,44,0.12)",
        line=dict(color="rgba(255,255,255,0)"),name="80% CI"))
fig.add_vline(x=today_str,line_dash="dash",line_color="#FF7F0E",line_width=2,annotation_text="Today",annotation_position="top right")
fig.update_layout(height=450,template="plotly_white",xaxis_title="Date",yaxis_title="Price (Rs)",legend=dict(orientation="h",y=1.02))
st.plotly_chart(fig, use_container_width=True)

cl,cr = st.columns(2)
with cl:
    st.subheader("Expected 30-Day Change")
    if "Expected_Change" in summary.columns:
        summ_sorted = summary.sort_values("Expected_Change",ascending=True)
        colors = ["#2CA02C" if v>=0 else "#D62728" for v in summ_sorted["Expected_Change"]]
        fig2 = go.Figure(go.Bar(x=summ_sorted["Expected_Change"],y=summ_sorted["Ticker"].str.replace(".NS",""),orientation="h",marker_color=colors))
        fig2.add_vline(x=0,line_color="black",line_width=1)
        fig2.update_layout(height=500,template="plotly_white",xaxis_title="Expected Change (%)")
        st.plotly_chart(fig2, use_container_width=True)
with cr:
    st.subheader("Forecast Summary")
    display_cols = [c for c in ["Ticker","Current_Price","Forecast_30d","Expected_Change","Direction"] if c in summary.columns]
    st.dataframe(summary[display_cols].sort_values("Expected_Change",ascending=False) if "Expected_Change" in summary.columns else summary[display_cols], use_container_width=True, hide_index=True)

st.subheader("Sentiment vs Forecast")
if not sent.empty and "Sentiment_Score" in sent.columns and "Expected_Change" in summary.columns:
    merged = summary.merge(sent[["Ticker","Sentiment_Score","Sentiment_Label"]],on="Ticker",how="left").dropna(subset=["Sentiment_Score","Expected_Change"])
    if not merged.empty:
        fig3 = px.scatter(merged,x="Sentiment_Score",y="Expected_Change",color="Expected_Change",color_continuous_scale=["#D62728","#FFFFFF","#2CA02C"],hover_data=["Ticker","Sentiment_Label"],text="Ticker",labels={"Sentiment_Score":"Sentiment Score","Expected_Change":"Expected Change (%)"})
        fig3.add_vline(x=50,line_dash="dash",line_color="gray",annotation_text="Neutral")
        fig3.add_hline(y=0,line_dash="dash",line_color="gray",annotation_text="No Change")
        fig3.update_traces(textposition="top center",textfont_size=7)
        fig3.update_layout(height=450,template="plotly_white")
        st.plotly_chart(fig3, use_container_width=True)
'''

with open(f"{APP_DIR}/pages/05_Price_Forecast.py", "w") as f:
    f.write(page5)
print("✅ Page 5 created")

# ── Page 6: Strategy Backtest ──
page6 = '''
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import sys
sys.path.append("/content/streamlit_app")
from data_loader import load_backtest_equity, load_backtest_trades, load_backtest_summary

st.set_page_config(page_title="Strategy Backtest", page_icon="📊", layout="wide")
st.markdown("""<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>""", unsafe_allow_html=True)
st.markdown("""<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">📊 Strategy Backtest — Signal vs Buy and Hold</h2></div>""", unsafe_allow_html=True)

equity  = load_backtest_equity()
trades  = load_backtest_trades()
summary = load_backtest_summary()
equity["Date"] = pd.to_datetime(equity["Date"])
if "Date" in trades.columns: trades["Date"] = pd.to_datetime(trades["Date"])

win_rate    = round((summary["Beat_Benchmark"]=="Yes").mean()*100,1) if not summary.empty and "Beat_Benchmark" in summary.columns else 0
med_outperf = round(summary["Outperformance_Pct"].median(),2) if "Outperformance_Pct" in summary.columns else 0
best_stock  = summary.nlargest(1,"Outperformance_Pct")["Ticker"].values[0].replace(".NS","") if not summary.empty else "N/A"

c1,c2,c3,c4 = st.columns(4)
c1.metric("Strategy Win Rate", f"{win_rate}%")
c2.metric("Median Outperformance", f"{med_outperf:+.2f}%")
c3.metric("Total Trades", len(trades))
c4.metric("Best Signal Stock", best_stock)

st.markdown("---")
st.sidebar.header("Controls")
ticker   = st.sidebar.selectbox("Select Stock", sorted(equity["Ticker"].unique()), index=0)
stock_eq = equity[equity["Ticker"]==ticker].sort_values("Date")

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=stock_eq["Date"],y=stock_eq["Strategy_Equity"],name="Signal Strategy",line=dict(color="#2CA02C",width=2.5)))
fig1.add_trace(go.Scatter(x=stock_eq["Date"],y=stock_eq["BuyHold_Equity"],name="Buy and Hold",line=dict(color="#1F77B4",width=1.5,dash="dash")))
fig1.add_hline(y=100000,line_dash="dot",line_color="gray",annotation_text="Starting Capital Rs1,00,000")
fig1.update_layout(height=400,template="plotly_white",xaxis_title="Date",yaxis_title="Portfolio Value (Rs)",legend=dict(orientation="h",y=1.02))
st.subheader(f"{ticker.replace('.NS','')} — Strategy vs Buy and Hold")
st.plotly_chart(fig1, use_container_width=True)

cl,cr = st.columns(2)
with cl:
    st.subheader("Outperformance by Stock")
    summ_sorted = summary.sort_values("Outperformance_Pct",ascending=True)
    colors = ["#2CA02C" if v>=0 else "#D62728" for v in summ_sorted["Outperformance_Pct"]]
    fig2 = go.Figure(go.Bar(x=summ_sorted["Outperformance_Pct"],y=summ_sorted["Ticker"].str.replace(".NS",""),orientation="h",marker_color=colors))
    fig2.add_vline(x=0,line_color="black",line_width=1)
    fig2.update_layout(height=500,template="plotly_white",xaxis_title="Outperformance (%)")
    st.plotly_chart(fig2, use_container_width=True)
with cr:
    st.subheader("Recent Trade Log")
    if not trades.empty:
        stock_trades = trades[trades["Ticker"]==ticker].sort_values("Date",ascending=False).head(20)
        if not stock_trades.empty:
            display_cols = [c for c in ["Date","Action","Price","Signal"] if c in stock_trades.columns]
            st.dataframe(stock_trades[display_cols], use_container_width=True, hide_index=True)
        else:
            st.info(f"No trades for {ticker}")
'''

with open(f"{APP_DIR}/pages/06_Strategy_Backtest.py", "w") as f:
    f.write(page6)
print("✅ Page 6 created")

# ── Page 7: Sentiment Intelligence ──
page7 = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.append("/content/streamlit_app")
from data_loader import load_sentiment, load_headlines

st.set_page_config(page_title="Sentiment Intelligence", page_icon="💬", layout="wide")
st.markdown("""<style>[data-testid="stSidebar"]{background:#F0F2F5;}</style>""", unsafe_allow_html=True)
st.markdown("""<div style="background:linear-gradient(135deg,#1F3864,#2E5EAA);padding:20px;border-radius:10px;color:white;margin-bottom:20px;"><h2 style="margin:0">💬 Sentiment Intelligence — FinBERT NLP</h2><p style="margin:5px 0 0 0;opacity:0.85">ProsusAI/FinBERT — transformer model trained on Bloomberg and Reuters financial news</p></div>""", unsafe_allow_html=True)

sent      = load_sentiment()
headlines = load_headlines()

c1,c2,c3,c4 = st.columns(4)
c1.metric("Headlines Analyzed", len(headlines) if not headlines.empty else 0)
c2.metric("Avg Confidence", f"{headlines[\'Confidence\'].mean():.3f}" if not headlines.empty and "Confidence" in headlines.columns else "N/A")
c3.metric("Most Bullish", sent.nlargest(1,"Sentiment_Score")["Ticker"].values[0].replace(".NS","") if not sent.empty else "N/A")
c4.metric("Most Bearish", sent.nsmallest(1,"Sentiment_Score")["Ticker"].values[0].replace(".NS","") if not sent.empty else "N/A")

st.sidebar.header("Controls")
ticker = st.sidebar.selectbox("Select Stock", ["All Stocks"]+sorted(sent["Ticker"].unique()), index=0)

st.markdown("---")
cl,cr = st.columns(2)
with cl:
    st.subheader("Sentiment Score by Stock")
    sent_sorted = sent.sort_values("Sentiment_Score",ascending=True)
    colors = ["#1A7A1A" if s>65 else "#2CA02C" if s>55 else "#FF7F0E" if s>45 else "#D62728" for s in sent_sorted["Sentiment_Score"]]
    fig1 = go.Figure(go.Bar(x=sent_sorted["Sentiment_Score"],y=sent_sorted["Ticker"].str.replace(".NS",""),orientation="h",marker_color=colors,text=sent_sorted["Sentiment_Score"].round(1),textposition="outside"))
    fig1.add_vline(x=50,line_dash="dash",line_color="black",annotation_text="Neutral (50)")
    fig1.update_layout(height=700,template="plotly_white",xaxis_title="Sentiment Score (0-100)",xaxis_range=[0,100])
    st.plotly_chart(fig1, use_container_width=True)
with cr:
    st.subheader("Headline Distribution")
    if not headlines.empty and "Label" in headlines.columns:
        label_counts = headlines["Label"].value_counts().reset_index()
        label_counts.columns = ["Label","Count"]
        label_colors = {"positive":"#2CA02C","negative":"#D62728","neutral":"#AAAAAA"}
        fig2 = px.pie(label_counts,names="Label",values="Count",color="Label",color_discrete_map=label_colors,hole=0.5)
        fig2.update_layout(height=300)
        st.plotly_chart(fig2, use_container_width=True)

    st.subheader("FinBERT Confidence")
    if not headlines.empty and "Confidence" in headlines.columns:
        fig3 = px.histogram(headlines,x="Confidence",nbins=30,color_discrete_sequence=["#1F77B4"])
        fig3.add_vline(x=headlines["Confidence"].mean(),line_dash="dash",line_color="#D62728",annotation_text=f"Mean: {headlines[\'Confidence\'].mean():.3f}")
        fig3.update_layout(height=280,template="plotly_white")
        st.plotly_chart(fig3, use_container_width=True)

st.subheader("Headlines with FinBERT Scores")
hl_display = headlines if ticker=="All Stocks" else headlines[headlines["Ticker"]==ticker] if not headlines.empty else pd.DataFrame()
if not hl_display.empty:
    display_cols = [c for c in ["Ticker","Headline","Label","Confidence","Polarity"] if c in hl_display.columns]
    st.dataframe(hl_display[display_cols].sort_values("Confidence",ascending=False).head(50) if "Confidence" in hl_display.columns else hl_display[display_cols], use_container_width=True, hide_index=True)
else:
    st.info("No headlines found")
'''

with open(f"{APP_DIR}/pages/07_Sentiment_Intelligence.py", "w") as f:
    f.write(page7)
print("✅ Page 7 created")

# ── requirements.txt ──
with open(f"{APP_DIR}/requirements.txt", "w") as f:
    f.write("streamlit==1.35.0\npandas==2.2.0\nnumpy==1.26.0\nplotly==5.22.0\n")
print("✅ requirements.txt created")

print("\n✅ ALL 7 PAGES CREATED SUCCESSFULLY")

✅ Page 2 created
✅ Page 3 created
✅ Page 4 created
✅ Page 5 created
✅ Page 6 created
✅ Page 7 created
✅ requirements.txt created

✅ ALL 7 PAGES CREATED SUCCESSFULLY


In [22]:
import subprocess
import threading
import time
import os
from pyngrok import ngrok

# Kill any old processes
os.system("pkill -f streamlit 2>/dev/null")
time.sleep(3)

def run():
    subprocess.run([
        "streamlit", "run",
        "/content/streamlit_app/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ])

thread = threading.Thread(target=run, daemon=True)
thread.start()

print("Starting Streamlit...")
time.sleep(10)


ngrok.set_auth_token("3GZI1gYzevRRUU33YtPWzH6Eliu_4PTafMhxJfk8RDdPALRce")
url = ngrok.connect(8501)

print(f"\n{'='*55}")
print(f"  🚀 APP IS LIVE!")
print(f"  URL: {url}")
print(f"{'='*55}")
print(f"\n  Open the URL and test all 7 pages")
print(f"  Share with anyone — works in any browser")

Starting Streamlit...



  🚀 APP IS LIVE!
  URL: NgrokTunnel: "https://porcupine-mossy-buzz.ngrok-free.dev" -> "http://localhost:8501"

  Open the URL and test all 7 pages
  Share with anyone — works in any browser


## ✅ Streamlit App Running!

### Test in Colab:
Run Cell 15 → get ngrok URL → open in browser

### Deploy Permanently to Streamlit Cloud (Phase 2):
1. Push streamlit_app/ folder to GitHub
2. Go to share.streamlit.io
3. Connect GitHub → select repo
4. Main file: app.py
5. Deploy → get permanent URL

### Your 7-page app covers:
1. app.py              → Home / Market Overview
2. 02_Stock_Deep_Dive  → Technical charts (Plotly)
3. 03_Fundamental      → Scores, ROE, grades
4. 04_Portfolio_Risk   → Allocation, Sharpe, VaR
5. 05_Price_Forecast   → Prophet predictions
6. 06_Strategy_Backtest→ Signal vs Buy & Hold
7. 07_Sentiment        → FinBERT NLP analysis